#  TrustCV v0.1: Advanced Workflow





## Universal Runner, Clinical Metrics & Regulatory Reporting



While `TrustCVValidator` is great for quick, high-level checks, advanced users often need more control. This notebook demonstrates how to use the core components of **TrustCV** individually to build a custom, regulatory-grade validation pipeline.

**We will cover:**

1. **`UniversalCVRunner`**: Running cross-validation loops with standardized logging and result capture.
2. **`ClinicalMetrics`**: Calculating detailed medical metrics (NNT, Likelihood Ratios) manually.
3. **`RegulatoryReport`**: Generating an HTML compliance report from your custom run.







In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine  # Using Wine as proxy for a multiclass/binary clinical task
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC
from sklearn.datasets import load_breast_cancer, make_classification

# TrustCV Imports
from trustcv.core import UniversalCVRunner
from trustcv.splitters import StratifiedKFold, RepeatedKFold
from trustcv.metrics import ClinicalMetrics
from trustcv.reporting import RegulatoryReport



## 1. Setup & Data Loading



We will use the **breast_cancer** dataset for a binary classification task.

In [2]:
data = load_breast_cancer()
X, y = data.data, data.target

# Create a small imbalanced subset for clearer visualization of LOOCV/LPOCV
X_small, y_small = make_classification(n_samples=25, n_classes=2, weights=[0.7, 0.3], random_state=42)

print(f"Full Dataset: {X.shape}, Class Balance: {np.bincount(y)}")
print(f"Small Dataset: {X_small.shape}, Class Balance: {np.bincount(y_small)}")

Full Dataset: (569, 30), Class Balance: [212 357]
Small Dataset: (25, 20), Class Balance: [18  7]


## 2. The Universal CV Runner



The `UniversalCVRunner` is a framework-agnostic engine. It standardizes the training loop, whether you use Scikit-Learn, PyTorch, or TensorFlow.

Here, we use a **Support Vector Machine (SVM)** to show how it handles probability calibration and complex pipelines.

### Class Distribution Logger

`ClassDistributionLogger` gives you the per-class percentages for both train and validation splits across folds. You can tailor label names or silence the messages by tweaking the callback parameters. 

**Suggested next step:** rerun your notebook cell to confirm the new distribution logs match your expectations.

In [3]:
from trustcv import ClassDistributionLogger


# 1. Define the Model (Using a Pipeline)
# Note: probability=True is needed for AUC/Likelihood Ratios
model = make_pipeline(
    StandardScaler(), 
    SVC(kernel='rbf', probability=True, random_state=42)
)

# 2. Define the Splitter
#cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv = RepeatedKFold(n_splits=5, n_repeats=2, random_state=42)

# 3. Initialize the Runner
# 'framework="auto"' will detect this is a scikit-learn model
runner = UniversalCVRunner(
    cv_splitter=cv, 
    framework="auto", 
    verbose=2  # Show progress bars, verbose=2 for detailed output, verbose=1 for less output, verbose=0 for none
)

# Optional: Add Callbacks (e.g., Logging)
class_logger = ClassDistributionLogger(
    labels=y,
    #label_names={0: "Benign", 1: "Malignant"},
    verbose=1,      # 0 = silent
    decimals=1
)


# 4. Run Validation
print(" Starting Universal Cross-Validation...")
results = runner.run(
    model=model, 
    data=(X, y), 
    callbacks=[class_logger]
)

# 5. Inspect Results
print("\n=== CV Results Summary ===")
print(results.summary())

 Starting Universal Cross-Validation...
Using sklearn adapter for cross-validation
Starting 10-fold cross-validation

Fold 1:
  Training samples: 455
  Validation samples: 114
  Train class distribution: 0: 169 (37.1%), 1: 286 (62.9%)
  Val class distribution:   0: 43 (37.7%), 1: 71 (62.3%)
Fold 1 completed
  score: 0.9825
  accuracy: 0.9825
  f1: 0.9861
  precision: 0.9726
  recall: 1.0000
  balanced_accuracy: 0.9767
  roc_auc: 0.9974

Fold 2:
  Training samples: 455
  Validation samples: 114
  Train class distribution: 0: 175 (38.5%), 1: 280 (61.5%)
  Val class distribution:   0: 37 (32.5%), 1: 77 (67.5%)
Fold 2 completed
  score: 0.9825
  accuracy: 0.9825
  f1: 0.9868
  precision: 1.0000
  recall: 0.9740
  balanced_accuracy: 0.9870
  roc_auc: 1.0000

Fold 3:
  Training samples: 455
  Validation samples: 114
  Train class distribution: 0: 169 (37.1%), 1: 286 (62.9%)
  Val class distribution:   0: 43 (37.7%), 1: 71 (62.3%)
Fold 3 completed
  score: 0.9737
  accuracy: 0.9737
  f1: 0.97

### Why use the Runner?



Unlike `cross_validate`, the `UniversalCVRunner` automatically:

- Captures **predictions** and **probabilities** for every fold (crucial for deep analysis).
- Stores trained **models** for inspection.
- Tracks split **indices** for reproducibility.

------



## 3. Deep Dive: Clinical Metrics



Standard metrics (Accuracy/F1) are often insufficient for medical devices. We need to know **"Number Needed to Treat" (NNT)** and **Likelihood Ratios**.

We will extract the predictions from the `runner` results and compute these metrics.

In [4]:
import numpy as np
from trustcv.metrics import ClinicalMetrics

y_true = [1, 0, 1, 1, 0, 0, 1, 0]
y_pred = [1, 0, 1, 0, 0, 1, 1, 0]

cm = ClinicalMetrics()  # add confidence_level=0.95 if you want CIs
#cm = ClinicalMetrics(confidence_level=0.95)
result = cm.calculate_all(y_true, y_pred, None)

#clin_metrics = ClinicalMetrics(confidence_level=0.95)
print(cm.format_report(result))

CLINICAL PERFORMANCE METRICS REPORT

PRIMARY DIAGNOSTIC METRICS
------------------------------
Sensitivity:  75.0% [30.1%, 95.4%]
Specificity:  75.0% [30.1%, 95.4%]
PPV:          75.0% [30.1%, 95.4%]
NPV:          75.0% [30.1%, 95.4%]

PERFORMANCE METRICS
------------------------------
Accuracy:         75.0%
Accuracy CI:      [40.9%, 92.9%]
F1 Score:         0.750

CLINICAL UTILITY
------------------------------
Youden's Index:     0.500
LR+:                3.00
LR-:                0.33
Diagnostic OR:      9.00
NNT:                n/a
NNS:                n/a
LR+ interpretation:  Small increase in post-test probability
LR- interpretation:  Small decrease in post-test probability
Diagnostic OR CI:   [0.4, 220.9]

CONFUSION MATRIX
------------------------------
True Positives:     3
True Negatives:     3
False Positives:    1
False Negatives:    1

CLINICAL SIGNIFICANCE
------------------------------
Screening Suitable:     No
Diagnostic Suitable:    No
Risk Stratification:    No

RECOMM

In [5]:
# Concatenate predictions from all folds to get a "global" view
# Note: This simulates how the model performs on the entire dataset when used as a test set.
all_y_true = []
all_y_pred = []
all_y_proba = []

# CVResults stores data per-fold. We reconstruct the full dataset.
# Note: We must be careful to align them correctly. The runner stores indices.
reconstructed_y = np.zeros(len(y))
reconstructed_pred = np.zeros(len(y))
reconstructed_proba = np.zeros(len(y))

for i, indices in enumerate(results.indices):
    train_idx, test_idx = indices
    
    # Get predictions for this fold
    # predictions[i] usually shaped (n_samples,) or (n_samples, n_classes)
    fold_preds = results.predictions[i]
    fold_probas = results.probabilities[i]
    
    # Handle binary classification shape issues if necessary
    if fold_probas.ndim > 1:
        fold_probas = fold_probas[:, 1] # Positive class probability
        
    reconstructed_y[test_idx] = y[test_idx]
    reconstructed_pred[test_idx] = fold_preds
    reconstructed_proba[test_idx] = fold_probas

print("✅ Reconstructed full cross-validated predictions.")

# --- Calculate Clinical Metrics ---
clin_metrics = ClinicalMetrics(confidence_level=0.95)
metrics_report = clin_metrics.calculate_all(
    y_true=reconstructed_y,
    y_pred=reconstructed_pred,
    y_proba=reconstructed_proba
)

# Display the rich report
print(clin_metrics.format_report(metrics_report))

✅ Reconstructed full cross-validated predictions.
CLINICAL PERFORMANCE METRICS REPORT

PRIMARY DIAGNOSTIC METRICS
------------------------------
Sensitivity:  98.3% [96.4%, 99.2%]
Specificity:  95.8% [92.1%, 97.8%]
PPV:          97.5% [95.3%, 98.7%]
NPV:          97.1% [93.9%, 98.7%]

PERFORMANCE METRICS
------------------------------
Accuracy:         97.4%
Accuracy CI:      [95.7%, 98.4%]
F1 Score:         0.979
AUC-ROC:          0.995 [0.989, 0.999]
Average Precision: 0.997
Optimal Threshold (Youden): 0.334
  ↳ Sensitivity @ Threshold: 99.4%
  ↳ Specificity @ Threshold: 95.8%

CLINICAL UTILITY
------------------------------
Youden's Index:     0.941
LR+:                23.16
LR-:                0.02
Diagnostic OR:      1319.50
NNT:                n/a
NNS:                n/a
LR+ interpretation:  Strong increase in post-test probability
LR- interpretation:  Strong decrease in post-test probability
Diagnostic OR CI:   [463.0, 3760.7]

CONFUSION MATRIX
------------------------------
Tru

## Using `oob_clinical_metrics` simpler function


 `oob_clinical_metrics` is a helper function designed to calculate clinical metrics using **aggregated out-of-bag predictions** from a cross-validation workflow.

Here is a detailed breakdown of its functionality:

### **Purpose**
Instead of averaging metrics calculated separately for each fold (like standard `cross_val_score`), this function:
1.  **Reconstructs the full dataset's predictions** by stitching together the predictions from every test fold (where the samples were "out-of-bag").
2.  **Calculates metrics globally** on these concatenated predictions/labels. This is particularly useful for metrics like **ROC-AUC**, **Sensitivity**, and **Specificity**, where a global calculation can differ from the average of fold-wise calculations, especially with imbalanced data or small fold sizes.

### **Parameters**
* `results`: A `CVResults` object (typically returned by `UniversalCVRunner`). It must contain the split indices and stored predictions/probabilities for each fold.
* `y`: The full target vector (true labels) aligned with the original dataset. This is sliced using the test indices to match the predictions.
* `clinical` (Optional): A custom `ClinicalMetrics` instance. If not provided, a default one is created.

### **How It Works**
1.  It iterates through the indices and scores stored in `results`.
2.  For each fold, it extracts the `y_true` (using test indices), `y_pred`, and `y_proba` (probabilities).
3.  It handles probability formatting (e.g., extracting the positive class column if a 2D array is provided).
4.  It concatenates these chunks into single arrays (`y_true_all`, `y_pred_all`, `y_proba_all`).
5.  It calls `ClinicalMetrics.calculate_all()` to compute metrics like Sensitivity, Specificity, PPV/NPV, and Likelihood Ratios on the aggregated data.



In [6]:
from trustcv.metrics import oob_clinical_metrics

metrics = oob_clinical_metrics(results, y)
print(metrics)
# Display the rich report

clin_metrics = ClinicalMetrics(confidence_level=0.95)
print(clin_metrics.format_report(metrics))

{'sensitivity': np.float64(0.9873949579831933), 'sensitivity_ci': (np.float64(0.9762189028661052), np.float64(0.9933545195765379)), 'specificity': np.float64(0.9599056603773585), 'specificity_ci': (np.float64(0.9367339138699331), np.float64(0.9748186997430612)), 'ppv': np.float64(0.9764542936288089), 'ppv_ci': (np.float64(0.962617303284332), np.float64(0.9852480902381368)), 'npv': np.float64(0.9783653846153846), 'npv_ci': (np.float64(0.9593998512132962), np.float64(0.988577037772262)), 'accuracy': np.float64(0.9771528998242531), 'accuracy_ci': (np.float64(0.9667337488492167), np.float64(0.9843615118900739)), 'f1_score': np.float64(0.9818941504178273), 'youdens_index': np.float64(0.9473006183605519), 'lr_positive': np.float64(24.626791893227868), 'lr_negative': np.float64(0.013131542543307203), 'diagnostic_odds_ratio': np.float64(1875.3921568627452), 'diagnostic_odds_ratio_ci': (828.3647428479605, 4245.829838110135), 'nnt': None, 'nns': None, 'auc_roc': 0.9946091644204852, 'auc_roc_ci':

# **ClinicalMetrics**


Here’s a concise **scientific breakdown of every metric** in your `ClinicalMetrics` output, including its **definition**, **formula**, and **clinical interpretation**.
Each metric originates from the **2×2 confusion matrix**:

|                         | **Predicted Positive**      | **Predicted Negative** |
| ----------------------- | --------------------------- | ---------------------- |
| **True Positive (TP)**  | Disease correctly detected  | –                      |
| **False Negative (FN)** | Missed disease              | –                      |
| **False Positive (FP)** | Healthy but wrongly flagged | –                      |
| **True Negative (TN)**  | Healthy correctly ruled out | –                      |

---

### 🔹 1. Sensitivity (a.k.a. Recall, True Positive Rate)

**Definition:** Fraction of actual positive cases correctly detected.
**Formula:** `TP / (TP + FN)`
**Clinical meaning:** Ability to detect the disease when it’s present.

* High sensitivity → fewer missed cases (good for screening tests).
  **CI:** 95% confidence interval for this estimate.

---

### 🔹 2. Specificity (True Negative Rate)

**Definition:** Fraction of actual negative cases correctly identified as negative.
**Formula:** `TN / (TN + FP)`
**Clinical meaning:** Ability to rule out disease in healthy individuals.

* High specificity → fewer false alarms (good for confirmatory tests).
  **CI:** 95% confidence interval for this estimate.

---

### 🔹 3. PPV (Positive Predictive Value)

**Definition:** Probability that a positive test result truly indicates disease.
**Formula:** `TP / (TP + FP)`
**Clinical meaning:** “If the test says positive, how likely is it real?”
**CI:** Confidence interval for PPV.

---

### 🔹 4. NPV (Negative Predictive Value)

**Definition:** Probability that a negative test truly indicates no disease.
**Formula:** `TN / (TN + FN)`
**Clinical meaning:** “If the test says negative, how confident are we the patient is healthy?”
**CI:** Confidence interval for NPV.

---

### 🔹 5. Accuracy

**Definition:** Overall fraction of correct predictions.
**Formula:** `(TP + TN) / (TP + TN + FP + FN)`
**Clinical meaning:** General correctness — but may be misleading in imbalanced data (e.g., rare diseases).
**CI:** Confidence interval for accuracy.

---

### 🔹 6. F1 Score

**Definition:** Harmonic mean of precision (PPV) and recall (sensitivity).
**Formula:** `2 × (PPV × Sensitivity) / (PPV + Sensitivity)`
**Clinical meaning:** Balances false negatives and false positives — used in model optimization where both errors matter.

---

### 🔹 7. Youden’s Index (J)

**Definition:** Unified diagnostic effectiveness index.
**Formula:** `Sensitivity + Specificity – 1`
**Clinical meaning:** Ranges from 0 to 1; higher = better test discriminability.

* J = 0 → useless test
* J = 1 → perfect test

---

### 🔹 8. LR⁺ (Positive Likelihood Ratio)

**Definition:** How much a positive result *increases* the odds of disease.
**Formula:** `Sensitivity / (1 – Specificity)`
**Clinical meaning:**

* > 10 → strong evidence to rule **in** disease
* ~5 → moderate
* <2 → weak

---

### 🔹 9. LR⁻ (Negative Likelihood Ratio)

**Definition:** How much a negative result *reduces* the odds of disease.
**Formula:** `(1 – Sensitivity) / Specificity`
**Clinical meaning:**

* <0.1 → strong evidence to rule **out** disease
* ~0.3 → moderate
* > 0.5 → weak

---

### 🔹 10. Diagnostic Odds Ratio (DOR)

**Definition:** Ratio of odds of positivity in diseased vs. non-diseased.
**Formula:** `(TP × TN) / (FP × FN)`
**Clinical meaning:**

* DOR = 1 → useless
* DOR > 10 → good diagnostic test
* DOR > 100 → excellent discrimination

---

### 🔹 11. NNT (Number Needed to Treat)

**Definition:** Number of patients who need treatment for one to benefit (used when outcome probabilities known).
**Clinical meaning:** Lower = more effective intervention.
In your result → `None` because not applicable to diagnostic models.

---

### 🔹 12. NNS (Number Needed to Screen)

**Definition:** Number of people who must be screened to prevent one adverse outcome.
Also `None` because no outcome data supplied.

---

### 🔹 13. Confusion Matrix

**Structure:**
`{'true_positives': TP, 'true_negatives': TN, 'false_positives': FP, 'false_negatives': FN}`
Shows the **raw counts** behind all other metrics.

---

### 🔹 14. Clinical Significance Block

**Keys:**

* `screening_suitable`: True if test has high sensitivity (≥0.9).
* `diagnostic_suitable`: True if both sensitivity and specificity are high (≥0.85).
* `risk_stratification_suitable`: True if F1 or PPV/NPV balanced in mid-range.
* `recommendations`: Suggestions for next steps (e.g., increase specificity for diagnostic use).

---

###  Summary example of interpretation for output

| Metric         | Value | Interpretation                                    |
| :------------- | :---- | :------------------------------------------------ |
| Sensitivity    | 0.75  | Detects 75% of diseased patients                  |
| Specificity    | 0.75  | Correctly excludes 75% of healthy patients        |
| PPV            | 0.75  | 3 of 4 positive predictions are true              |
| NPV            | 0.75  | 3 of 4 negative predictions are true              |
| Accuracy       | 0.75  | 75% overall correctness                           |
| F1             | 0.75  | Balanced precision/recall performance             |
| Youden’s Index | 0.5   | Moderate diagnostic quality                       |
| LR⁺            | 3.0   | Positive result moderately increases disease odds |
| LR⁻            | 0.33  | Negative result moderately decreases disease odds |
| DOR            | 9.0   | Fair diagnostic discriminability                  |

---




## 4. Generating a Regulatory Report



Finally, we generate a **compliance-ready HTML report**. This is useful for FDA/CE technical files or internal audits.

In [11]:
from trustcv.reporting import UniversalRegulatoryReport

report_path = UniversalRegulatoryReport.from_runner(
    runner_results=results,
    model=model,
    data=(X, y),
    clinical_metrics=metrics,          # Add the Deep Clinical Metrics we just calculated
    output_path="reports/breast_cancer_regulatory2.pdf",
    report_format="pdf",
    model_name="Breast Cancer RF",
    manufacturer="Your Lab",
    intended_use="Assist clinicians in classifying breast cancer biopsies.",
)
print("Report:", report_path)

Report: reports/breast_cancer_regulatory2.pdf


### Clinical Performance Report

---

### **1. Executive Summary & Primary Metrics**
This section provides a high-level snapshot of the model's diagnostic capability. It is the first thing auditors or clinical leads will look at.

* **Sensitivity (Recall):** The percentage of actual positive cases (e.g., cancer) correctly identified by the model.
    * *Why it matters:* In medical screening, missing a disease (False Negative) is dangerous. High sensitivity means fewer missed cases.
* **Specificity:** The percentage of actual negative cases (e.g., healthy) correctly identified as negative.
    * *Why it matters:* Low specificity leads to "False Alarms" (False Positives), causing unnecessary anxiety and expensive follow-up tests.
* **PPV (Precision):** If the model predicts "Positive," how likely is it to be true?
* **NPV:** If the model predicts "Negative," how safe is the patient?

### **2. Dataset Characteristics**
Transparency about the data is a regulatory requirement (FDA/CE MDR). This section proves the model was tested on a relevant population.

* **Total Patients/Samples:** Confirms the study size was statistically meaningful.
* **Features:** The number of input variables (e.g., 30 biopsy measurements) used for prediction.
* **Class Distribution:** Shows the ratio of Healthy (0) vs. Disease (1).
    * *Why it matters:* An imbalanced dataset (e.g., 99% healthy) can fake high accuracy. Showing the count (212 vs 357) proves the test set was challenging enough.

### **3. Validation Methodology**
This section establishes the scientific rigor of your evaluation.

* **Method:** Explicitly names the strategy (e.g., `RepeatedKFold`, `StratifiedKFoldMedical`).
* **Number of Folds:** Tells the reader how many times the validation was repeated to ensure stability (e.g., 10 folds).
* **Mean Accuracy:** The average performance across all folds, plus the standard deviation ($\pm 0.010$).
    * *Why it matters:* A low standard deviation ($\pm$) proves the model is stable and robust, not just lucky on one specific data split.

### **4. Clinical Utility**
Standard ML metrics (Accuracy) don't tell doctors if a test is useful. This section translates math into medical value.

* **AUC-ROC:** The overall ability of the model to distinguish between classes at *any* threshold. (0.5 = Random Guess, 1.0 = Perfect).
* **Likelihood Ratios (LR+ / LR-):**
    * **LR+ (>10):** A positive result strongly rules *in* the disease.
    * **LR- (<0.1):** A negative result strongly rules *out* the disease.
    * *Interpretation:* Your report automatically interprets these values (e.g., "Strong increase in post-test probability").
* **Youden's Index:** The optimal cut-off point that maximizes both sensitivity and specificity.

### **5. Confusion Matrix**
The raw "truth table" of your model's decisions.

* **True Positives (TP):** Sick people correctly diagnosed. (Goal: Maximize).
* **True Negatives (TN):** Healthy people correctly reassured.
* **False Positives (FP):** Healthy people wrongly alarmed. (Cost: Anxiety, biopsy).
* **False Negatives (FN):** Sick people wrongly sent home. (Cost: Missed treatment, danger).
    * *Why it matters:* This allows auditors to verify your Sensitivity/Specificity calculations manually.

### **6. Clinical Significance & Recommendations**
This is the "Actionable Advice" engine of TrustCV.

* **Suitability Check:** Automatically flags if the model is safe for "Screening" (requires high sensitivity) or "Diagnosis" based on thresholds.
* **Recommendations:** Dynamic text generated based on your metrics.
    * *Example:* "High sensitivity (>95%) - suitable for screening applications."
    * *Why it matters:* It guides the end-user (clinician or deployer) on **how** to safely use this AI model in a real hospital workflow.

In [ ]:


# 2) Styled Clinical Performance Report (new)
clinical_report_path = UniversalRegulatoryReport.clinical_report_from_runner(
    runner_results=results,
    model=model,
    data=(X, y),
    clinical_metrics=metrics,   # pass metrics or let TrustCV recompute if predictions exist
    output_path="reports/breast_cancer_clinical1.html",
    report_format="html",        # or "html"
    model_name="Breast Cancer RF",
    manufacturer="Your Lab",
    intended_use="Assist clinicians in classifying breast cancer biopsies.",
)
print("Clinical performance report:", clinical_report_path)




Regulatory report: reports/breast_cancer_regulatory2.pdf
Clinical performance report: reports/breast_cancer_clinical1.html
